In [ ]:
import pandas as pd 
import time
import math
import numpy as np

In [2]:
def fetch_football_data(seasons, leagues):
    all_data = []
    
    for season in seasons:
        for league in leagues:
            url = f"https://www.football-data.co.uk/mmz4281/{season}/{league}.csv"
            
            try:
                df =pd.read_csv(url)
                df['Season'] = season   
                df['League'] = league
                all_data.append(df)
                
            except Exception as e:
                print(f"Error: {league} {season}: {e}") 
    final_df = pd.concat(all_data, ignore_index=True)
    return final_df


In [3]:
seasons = ['2122', '2223', '2324', '2425', '2526']
leagues = ['E0', 'SP1', 'I1', 'D1', 'F1']

big_df = fetch_football_data(seasons, leagues)


print(f"downloaded matches: {len(big_df)} ")
big_df.to_csv('../data/top5_leagues_raw_combined.csv', index=False)

C:\Users\huber\AppData\Local\Temp\ipykernel_8252\2147237358.py:10: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['Season'] = season
C:\Users\huber\AppData\Local\Temp\ipykernel_8252\2147237358.py:11: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['League'] = league


downloaded matches: 8447 


In [4]:
big_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 8447 entries, 0 to 8446
Columns: 164 entries, Div to LBCA
dtypes: float64(152), int64(2), str(10)
memory usage: 10.6 MB


In [5]:
big_df.isnull().sum().sort_values(ascending=False).head(20)

LBCA      7423
LBCD      7423
CLCA      7423
LBCH      7423
CLCH      7423
CLCD      7423
CLA       7402
CLD       7402
LBD       7402
LBH       7402
LBA       7402
CLH       7402
BMGMCA    7156
BVCH      7156
BFDCH     7156
BVA       7156
BFDH      7156
BMGMH     7156
BFDCD     7156
BFDCA     7156
dtype: int64

In [6]:
my_features = [
    'Date', 'HomeTeam', 'AwayTeam', 
    'FTHG', 'FTAG', 'FTR', 
    'HTHG', 'HTAG', 'HTR',
    'HS', 'AS', 'HST', 'AST', 
    'HR', 'AR', 
    'B365H', 'B365D', 'B365A',
    'Season', 'League'
]

In [7]:
big_df_clean = big_df[my_features].copy()

In [8]:
missing_values_clean = big_df_clean.isnull().mean() *100
print("percentage of missing values in cleaned dataset:")
print(missing_values_clean)

percentage of missing values in cleaned dataset:
Date        0.000000
HomeTeam    0.000000
AwayTeam    0.000000
FTHG        0.000000
FTAG        0.000000
FTR         0.000000
HTHG        0.011839
HTAG        0.011839
HTR         0.011839
HS          0.011839
AS          0.011839
HST         0.011839
AST         0.011839
HR          0.011839
AR          0.011839
B365H       0.011839
B365D       0.011839
B365A       0.011839
Season      0.000000
League      0.000000
dtype: float64


In [9]:
big_df_clean = big_df_clean.dropna()

In [10]:
rename_mapping = {
    'HomeTeam': 'home_team',
    'AwayTeam': 'away_team',
    'FTHG': 'home_goals',
    'FTAG': 'away_goals',
    'FTR': 'match_result',
    'HS': 'home_shots',
    'AS': 'away_shots',
    'HST': 'home_shots_target',
    'AST': 'away_shots_target',
    'HR': 'home_red_cards',
    'AR': 'away_red_cards',
    'HTHG': 'ht_home_goals',
    'HTAG': 'ht_away_goals',
    'HTR': 'ht_result'
}

big_df_clean.rename(columns=rename_mapping, inplace=True)

In [11]:
big_df_clean['Date'] = pd.to_datetime(big_df_clean['Date'], dayfirst=True)

In [12]:
big_df_clean = big_df_clean.sort_values('Date')
big_df_clean = big_df_clean.reset_index(drop=True)

In [13]:
big_df_clean['home_team_goals_avg_last_5'] = big_df_clean.groupby('home_team')['home_goals'].transform(lambda x: x.rolling(window=5, min_periods=1).mean().shift(1))
big_df_clean['away_goals_avg_last_5'] = big_df_clean.groupby('away_team')['away_goals'].transform(lambda x: x.rolling(window=5, min_periods=1).mean().shift(1))
big_df_clean['ht_home_goals_avg_last_5'] = big_df_clean.groupby('home_team')['ht_home_goals'].transform(lambda x: x.rolling(window=5, min_periods=1).mean().shift(1))
big_df_clean['ht_away_goals_avg_last_5'] = big_df_clean.groupby('away_team')['ht_away_goals'].transform(lambda x: x.rolling(window=5, min_periods=1).mean().shift(1))

In [14]:
big_df_clean['home_balance_1h'] = big_df_clean['ht_home_goals'] - big_df_clean['ht_away_goals']
big_df_clean['home_balance_2h'] = (big_df_clean['home_goals'] - big_df_clean['ht_home_goals']) - (big_df_clean['away_goals'] - big_df_clean['ht_away_goals'])
big_df_clean['away_balance_1h'] = -big_df_clean['home_balance_1h']
big_df_clean['away_balance_2h'] = -big_df_clean['home_balance_2h']

big_df_clean['home_form_1h_last_5'] = big_df_clean.groupby('home_team')['home_balance_1h'].transform(lambda x: x.rolling(5, min_periods=1).mean().shift(1))
big_df_clean['away_form_1h_last_5'] = big_df_clean.groupby('away_team')['away_balance_1h'].transform(lambda x: x.rolling(5, min_periods=1).mean().shift(1))
big_df_clean['home_form_2h_last_5'] = big_df_clean.groupby('home_team')['home_balance_2h'].transform(lambda x: x.rolling(window=5, min_periods=1).mean().shift(1))
big_df_clean['away_form_2h_last_5'] = big_df_clean.groupby('away_team')['away_balance_2h'].transform(lambda x: x.rolling(window=5, min_periods=1).mean().shift(1))

In [15]:

big_df_clean['home_shots_avg_last_5'] = big_df_clean.groupby('home_team')['home_shots'].transform(lambda x: x.rolling(window=5, min_periods=1).mean().shift(1))
big_df_clean['away_shots_avg_last_5'] = big_df_clean.groupby('away_team')['away_shots'].transform(lambda x: x.rolling(window=5, min_periods=1).mean().shift(1))

big_df_clean['home_shots_target_avg_last_5'] = big_df_clean.groupby('home_team')['home_shots_target'].transform(lambda x: x.rolling(window=5, min_periods=1).mean().shift(1))
big_df_clean['away_shots_target_avg_last_5'] = big_df_clean.groupby('away_team')['away_shots_target'].transform(lambda x: x.rolling(window=5, min_periods=1).mean().shift(1))


big_df_clean['home_red_cards_avg_last_5'] = big_df_clean.groupby('home_team')['home_red_cards'].transform(lambda x: x.rolling(window=5, min_periods=1).mean().shift(1))
big_df_clean['away_red_cards_avg_last_5'] = big_df_clean.groupby('away_team')['away_red_cards'].transform(lambda x: x.rolling(window=5, min_periods=1).mean().shift(1))


big_df_clean['home_goals_conceded_avg_last_5'] = big_df_clean.groupby('home_team')['away_goals'].transform(lambda x: x.rolling(window=5, min_periods=1).mean().shift(1))
big_df_clean['away_goals_conceded_avg_last_5'] = big_df_clean.groupby('away_team')['home_goals'].transform(lambda x: x.rolling(window=5, min_periods=1).mean().shift(1))


big_df_clean['target'] = big_df_clean['match_result'].map({'H': 2, 'D': 1, 'A': 0})
big_df_clean['home_points'] = big_df_clean['match_result'].map({'H': 3, 'D': 1, 'A': 0})
big_df_clean['away_points'] = big_df_clean['match_result'].map({'H': 0, 'D': 1, 'A': 3})

big_df_clean['home_points_avg_last_5'] = big_df_clean.groupby('home_team')['home_points'].transform(lambda x: x.rolling(window=5, min_periods=1).mean().shift(1))
big_df_clean['away_points_avg_last_5'] = big_df_clean.groupby('away_team')['away_points'].transform(lambda x: x.rolling(window=5, min_periods=1).mean().shift(1))

In [16]:
df_home_points = big_df_clean[['Date', 'home_team', 'home_points']].copy()
df_away_points = big_df_clean[['Date', 'away_team', 'away_points']].copy()
df_home_points.columns = ['Date', 'Team', 'Points']
df_away_points.columns = ['Date', 'Team', 'Points']
concatenated_points = pd.concat([df_home_points, df_away_points], ignore_index=True)
concatenated_points_sorted = concatenated_points.sort_values(by=['Team', 'Date'])
concatenated_points_sorted['overall_points_last_5'] = concatenated_points_sorted.groupby('Team')['Points'].transform(lambda x: x.rolling(window=5, min_periods=1).mean().shift(1))

big_df_clean = big_df_clean.merge(
    concatenated_points_sorted[['Date', 'Team', 'overall_points_last_5']],
    left_on=['Date', 'home_team'],
    right_on=['Date', 'Team'],
    how='left'
)
big_df_clean = big_df_clean.drop(columns=['Team'])
big_df_clean = big_df_clean.rename(columns={'overall_points_last_5': 'home_overall_points_last_5'})

big_df_clean = big_df_clean.merge(
    concatenated_points_sorted[['Date', 'Team', 'overall_points_last_5']],
    left_on=['Date', 'away_team'],
    right_on=['Date', 'Team'],
    how='left'
)
big_df_clean = big_df_clean.drop(columns=['Team'])
big_df_clean = big_df_clean.rename(columns={'overall_points_last_5': 'away_overall_points_last_5'})

In [25]:
sql_query = "CREATE TABLE matches (\n    id SERIAL PRIMARY KEY,\n"

for col, dtype in big_df_clean.dtypes.items():
    if "datetime" in str(dtype):
        sql_type = "DATE"
    elif "float" in str(dtype):
        sql_type = "FLOAT"
    elif "int" in str(dtype):
        sql_type = "INTEGER"
    else:
        sql_type = "TEXT"
        
    sql_query += f"    {col} {sql_type},\n"

sql_query = sql_query.rstrip(",\n") + "\n);"

with open("create_tablesql", "w") as file:
    file.write(sql_query)

In [26]:
import os
from dotenv import load_dotenv
from supabase import create_client, Client

load_dotenv(override=True)
url = os.environ.get("SUPABASE_URL").strip()
key = os.environ.get("SUPABASE_KEY").strip()

supabase: Client = create_client(url, key)

big_df_clean.columns = big_df_clean.columns.str.lower()
big_df_clean['date'] = big_df_clean['date'].astype(str)
big_df_clean = big_df_clean.replace({np.nan: None})

data_to_upload = big_df_clean.to_dict(orient='records')

batch_size = 1000
total_batches = math.ceil(len(data_to_upload) / batch_size)
print(f"starting upload to supabase: {len(data_to_upload)} matches in {total_batches} batches")

for i in range(total_batches):
    start_idx = i * batch_size
    end_idx = start_idx + batch_size
    batch = data_to_upload[start_idx:end_idx]

    response = supabase.table('matches').insert(batch).execute()
    print(f"Batch {i+1}/{total_batches} uploaded")

print("Data upload completed!")

starting upload to supabase: 8445 matches in 9 batches
Batch 1/9 uploaded
Batch 2/9 uploaded
Batch 3/9 uploaded
Batch 4/9 uploaded
Batch 5/9 uploaded
Batch 6/9 uploaded
Batch 7/9 uploaded
Batch 8/9 uploaded
Batch 9/9 uploaded
Data upload completed!
